<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/W5_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import random
import yaml
from torch.utils.data import Dataset, DataLoader

In [2]:
!pip install -q transformers accelerate sentencepiece safetensors pyyaml

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install transformers
!pip install accelerate
!pip install peft
!pip install huggingface_hub
!pip install pyyaml

In [5]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Total 371 (delta 0), reused 0 (delta 0), pack-reused 371 (from 1)
Receiving objects: 100% (371/371), 9.35 MiB | 27.42 MiB/s, done.
Resolving deltas: 100% (159/159), done.


In [6]:
%cd /content/Kronos

/content/Kronos


In [7]:
!ls

examples  finetune	LICENSE  README.md	   tests
figures   finetune_csv	model	 requirements.txt  webui


In [8]:
df = pd.read_csv(
"/content/drive/MyDrive/PRISM/processed/RELIANCE.NS_processed.csv")
df.head()

,Date,Close,High,Low,Open,Volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target
0,2019-04-02,615.133118,621.064466,610.883802,618.807021,17526740,-0.001545,-0.001546,607.125757,603.150867,587.337653,0.016157,-0.206535,21556498.85,0.003775,1.569316,-0.010434
1,2019-04-03,608.714844,621.020165,607.298386,616.483114,17169813,-0.010434,-0.010489,607.829541,604.264093,590.638626,0.016430,-0.020365,21548509.20,-0.005912,1.593855,-0.016107
2,2019-04-04,598.910339,612.477168,596.342998,610.396773,18320845,-0.016107,-0.016238,608.165942,603.223895,593.192639,0.017118,0.067038,21685676.45,-0.003946,1.623214,0.000628
3,2019-04-05,599.286560,603.712925,594.461833,602.407152,14717266,0.000628,0.000628,607.625916,602.270007,595.164584,0.016639,-0.196693,21104925.95,0.005859,1.608380,-0.018207
4,2019-04-08,588.375610,600.880142,585.919014,600.216205,19081843,-0.018207,-0.018374,602.084094,601.716705,596.470364,0.017331,0.296562,21172113.50,-0.005267,1.638698,0.003912


In [9]:
df = df.rename(columns={
    "Date":"timestamps",
    "Open":"open",
    "High":"high",
    "Low":"low",
    "Close":"close",
    "Volume":"volume"
})
df["amount"] = 0

In [10]:
df["timestamps"] = pd.to_datetime(df["timestamps"])
df.tail()

,timestamps,close,high,low,open,volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target,amount
1412,2024-12-23,1211.834717,1216.692666,1202.812534,1204.597171,10052824,0.014104,0.014006,1220.777417,1241.647192,1266.351343,0.012267,-0.505101,14831684.80,0.007035,1.320625,0.000368,0
1413,2024-12-24,1212.280762,1222.988340,1210.545745,1211.834663,6734917,0.000368,0.000368,1216.306030,1235.490369,1262.735071,0.012095,-0.330047,14706052.80,-0.001086,1.337080,-0.005070,0
1414,2024-12-26,1206.133911,1217.188348,1203.853555,1213.767935,10016178,-0.005070,-0.005083,1209.028882,1229.378162,1258.935388,0.012103,0.487201,14728224.40,0.000950,1.332965,0.003699,0
1415,2024-12-27,1210.595459,1217.386785,1206.580087,1207.869004,7000397,0.003699,0.003692,1207.165015,1225.229004,1256.469189,0.011698,-0.301091,14326873.65,0.002661,1.295769,-0.008476,0
1416,2024-12-30,1200.333984,1212.726960,1197.756270,1205.985254,8818766,-0.008476,-0.008513,1208.235767,1219.067212,1252.429083,0.010903,0.259752,14107696.60,-0.007076,1.294135,0.003923,0


In [11]:
save_path="/content/Kronos/data"
os.makedirs(save_path,exist_ok=True)
df.to_csv(
f"{save_path}/reliance.csv",
index=False)

In [12]:
pd.read_csv(
"/content/Kronos/data/reliance.csv").head()

,timestamps,close,high,low,open,volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target,amount
0,2019-04-02,615.133118,621.064466,610.883802,618.807021,17526740,-0.001545,-0.001546,607.125757,603.150867,587.337653,0.016157,-0.206535,21556498.85,0.003775,1.569316,-0.010434,0
1,2019-04-03,608.714844,621.020165,607.298386,616.483114,17169813,-0.010434,-0.010489,607.829541,604.264093,590.638626,0.016430,-0.020365,21548509.20,-0.005912,1.593855,-0.016107,0
2,2019-04-04,598.910339,612.477168,596.342998,610.396773,18320845,-0.016107,-0.016238,608.165942,603.223895,593.192639,0.017118,0.067038,21685676.45,-0.003946,1.623214,0.000628,0
3,2019-04-05,599.286560,603.712925,594.461833,602.407152,14717266,0.000628,0.000628,607.625916,602.270007,595.164584,0.016639,-0.196693,21104925.95,0.005859,1.608380,-0.018207,0
4,2019-04-08,588.375610,600.880142,585.919014,600.216205,19081843,-0.018207,-0.018374,602.084094,601.716705,596.470364,0.017331,0.296562,21172113.50,-0.005267,1.638698,0.003912,0


In [13]:
config={
"data":{
"data_path":"/content/Kronos/data/reliance.csv",
"lookback_window":30,
"predict_window":5,
"max_context":30
},

"training":{
"batch_size":32,
"epochs":5,
"learning_rate":1e-4
},

"model_paths":{
"pretrained_tokenizer":"NeoQuasar/Kronos-Tokenizer-base",
"pretrained_predictor":"NeoQuasar/Kronos-base",
"base_save_path":"/content/Kronos/checkpoints",
"exp_name":"Reliance"
}
}

In [14]:
with open(
"/content/Kronos/config.yaml",
"w"
) as f:
    yaml.dump(config,f)

In [15]:
!cat /content/Kronos/config.yaml

data:
  data_path: /content/Kronos/data/reliance.csv
  lookback_window: 30
  max_context: 30
  predict_window: 5
model_paths:
  base_save_path: /content/Kronos/checkpoints
  exp_name: Reliance
  pretrained_predictor: NeoQuasar/Kronos-base
  pretrained_tokenizer: NeoQuasar/Kronos-Tokenizer-base
training:
  batch_size: 32
  epochs: 5
  learning_rate: 0.0001


In [16]:
os.path.exists(
"/content/Kronos/config.yaml")

True

In [17]:
from finetune_csv.config_loader import CustomFinetuneConfig

In [18]:
config=CustomFinetuneConfig(
"/content/Kronos/config.yaml")